# Rare Earth Extraction: Liquid-Liquid Separation

This notebook demonstrates **liquid-liquid extraction (LLE)** for separating rare earth elements (REE) using the difflow framework.

## Background

Rare earth elements are critical materials for:
- **Permanent magnets** (Nd, Dy) - electric vehicles, wind turbines
- **Electronics** - smartphones, displays
- **Clean energy** - batteries, catalysts

Solvent extraction is the primary industrial method for REE separation, using extractants like **D2EHPA** (di-2-ethylhexyl phosphoric acid).

## Elements in This Example

| Element | Symbol | Type | K-value | Application |
|---------|--------|------|---------|-------------|
| Lanthanum | La | Light REE | 0.5 | Catalysts |
| Neodymium | Nd | Light REE | 2.0 | Magnets |
| Dysprosium | Dy | Heavy REE | 8.0 | High-temp magnets |

In [ ]:
import jax
import jax.numpy as jnp
from jax import grad, jacfwd

jax.config.update("jax_enable_x64", True)

from difflow.streams import make_stream, get_flows
from difflow.units.lle import (
    MultistageCascade,
    CascadeParams,
    DifferentialContactor,
    ContactorParams,
    LLEEquilibrium,
    DistributionCoeffs,
    separation_factor,
)

## 1. Define Distribution Coefficients

The **distribution coefficient** K determines how a solute partitions between phases:

$$K = \frac{[\text{REE}]_{organic}}{[\text{REE}]_{aqueous}}$$

For D2EHPA extraction:
- K increases with atomic number (heavier REE extract preferentially)
- K decreases with temperature (extraction is exothermic)

Temperature dependence follows van't Hoff:
$$K(T) = K_0 \exp\left(-\frac{\Delta H}{R}\left(\frac{1}{T} - \frac{1}{T_{ref}}\right)\right)$$

In [ ]:
# Distribution coefficients at reference temperature (25°C)
K_La = 0.5   # Light REE, low extraction
K_Nd = 2.0   # Medium - primary target
K_Dy = 8.0   # Heavy REE, high extraction

# Temperature dependence (extraction is exothermic, dH < 0)
dH_La = -15000.0  # J/mol
dH_Nd = -18000.0
dH_Dy = -22000.0

dist_coeffs = DistributionCoeffs(
    species=("La", "Nd", "Dy"),
    K0=(K_La, K_Nd, K_Dy),
    dH=(dH_La, dH_Nd, dH_Dy),
    Tref=298.15,
)

# Create LLE equilibrium calculator
lle_eq = LLEEquilibrium(
    solutes=["La", "Nd", "Dy"],
    aqueous_carrier="H2O",
    organic_carrier="Organic",
    K_coeffs=dist_coeffs,
    activity_model="K",
)

print("Distribution coefficients at 25°C:")
print(f"  K_La = {K_La:.2f}  (extracts poorly)")
print(f"  K_Nd = {K_Nd:.2f}  (moderate)")
print(f"  K_Dy = {K_Dy:.2f}  (extracts well)")

print(f"\nSeparation factors (higher = easier separation):")
print(f"  SF(Nd/La) = {separation_factor(K_Nd, K_La):.2f}")
print(f"  SF(Dy/Nd) = {separation_factor(K_Dy, K_Nd):.2f}")
print(f"  SF(Dy/La) = {separation_factor(K_Dy, K_La):.2f}")

## 2. Define Feed Streams

We model a typical REE leach solution:
- **Aqueous feed**: Dissolved REE ions in water
- **Organic solvent**: D2EHPA in kerosene

In [ ]:
# Aqueous feed: REE leach solution
# Typical concentrations: ~1-3 g/L per element
feed = make_stream(
    flows={
        "H2O": 55.5,    # ~1 L/s of water
        "La": 0.01,     # ~1.4 g/L
        "Nd": 0.02,     # ~2.9 g/L (main target)
        "Dy": 0.005,    # ~0.8 g/L
    },
    T=298.15,
    P=101325.0,
)

# Organic solvent: D2EHPA in kerosene
solvent = make_stream(
    flows={
        "Organic": 10.0,
        "La": 0.0,
        "Nd": 0.0,
        "Dy": 0.0,
    },
    T=298.15,
    P=101325.0,
)

feed_flows = get_flows(feed)
print("Aqueous feed (mol/s):")
print(f"  H2O: {feed_flows['H2O']:.2f}")
print(f"  La:  {feed_flows['La']:.4f}")
print(f"  Nd:  {feed_flows['Nd']:.4f}")
print(f"  Dy:  {feed_flows['Dy']:.4f}")

print(f"\nOrganic solvent: {get_flows(solvent)['Organic']:.2f} mol/s")

## 3. Multi-Stage Cascade Extraction

A **counter-current cascade** is the most efficient configuration:
- Fresh solvent contacts the most depleted aqueous
- Fresh aqueous contacts the most loaded solvent

```
Feed →  [1] → [2] → [3] → [4] → [5] → Raffinate
              ↑     ↑     ↑     ↑     ↑
Extract ← [1] ← [2] ← [3] ← [4] ← [5] ← Solvent
```

The **Kremser equation** gives the analytical solution for linear equilibria.

In [ ]:
cascade_params = CascadeParams(
    n_stages=5,
    equilibrium=lle_eq,
    flow_config="counter_current",
)
cascade = MultistageCascade(cascade_params)

raffinate, extract, info = cascade(feed, solvent, T=298.15)

raff_flows = get_flows(raffinate)
ext_flows = get_flows(extract)

print(f"Counter-current cascade with {cascade_params.n_stages} stages")
print("\nRaffinate (aqueous outlet):")
print(f"  La: {float(raff_flows['La']):.6f} mol/s")
print(f"  Nd: {float(raff_flows['Nd']):.6f} mol/s")
print(f"  Dy: {float(raff_flows['Dy']):.6f} mol/s")

print(f"\nExtract (organic outlet):")
print(f"  La: {float(ext_flows['La']):.6f} mol/s")
print(f"  Nd: {float(ext_flows['Nd']):.6f} mol/s")
print(f"  Dy: {float(ext_flows['Dy']):.6f} mol/s")

# Calculate recoveries
rec_La = float(ext_flows['La']) / feed_flows['La'] * 100
rec_Nd = float(ext_flows['Nd']) / feed_flows['Nd'] * 100
rec_Dy = float(ext_flows['Dy']) / feed_flows['Dy'] * 100

print(f"\n📊 Recoveries to extract:")
print(f"  La: {rec_La:5.1f}%  {'█' * int(rec_La/5)}")
print(f"  Nd: {rec_Nd:5.1f}%  {'█' * int(rec_Nd/5)}")
print(f"  Dy: {rec_Dy:5.1f}%  {'█' * int(rec_Dy/5)}")

## 4. Effect of Number of Stages

More stages → higher recovery, but diminishing returns.

The differentiable model allows us to compute ∂Recovery/∂N analytically!

In [ ]:
print("Effect of Number of Stages:")
print(f"{'Stages':>8} {'Nd Rec%':>10} {'La Rec%':>10} {'Dy Rec%':>10}")
print("-" * 40)

for n in [1, 2, 3, 5, 7, 10]:
    params = CascadeParams(
        n_stages=n,
        equilibrium=lle_eq,
        flow_config="counter_current",
    )
    cascade_fn = MultistageCascade(params)
    
    _, extract, _ = cascade_fn(feed, solvent, T=298.15)
    ext_flows = get_flows(extract)
    
    nd_rec = float(ext_flows['Nd']) / feed_flows['Nd'] * 100
    la_rec = float(ext_flows['La']) / feed_flows['La'] * 100
    dy_rec = float(ext_flows['Dy']) / feed_flows['Dy'] * 100
    
    print(f"{n:>8} {nd_rec:>10.1f} {la_rec:>10.1f} {dy_rec:>10.1f}")

## 5. Sensitivity Analysis with Automatic Differentiation

How sensitive is Nd recovery to operating parameters?

We compute exact gradients using JAX's automatic differentiation.

In [ ]:
def nd_recovery(n_stages: float, S_F_ratio: float, T: float) -> float:
    """Calculate Nd recovery to extract."""
    solvent_adj = make_stream(
        flows={
            "Organic": 10.0 * S_F_ratio,
            "La": 0.0, "Nd": 0.0, "Dy": 0.0,
        },
        T=T,
        P=101325.0,
    )
    
    params = CascadeParams(
        n_stages=n_stages,
        equilibrium=lle_eq,
        flow_config="counter_current",
    )
    cascade_fn = MultistageCascade(params)
    
    _, extract, _ = cascade_fn(feed, solvent_adj, T=T)
    ext_flows = get_flows(extract)
    
    return ext_flows['Nd'] / feed_flows['Nd']


# Base case
n_stages_val = 5.0
SF_ratio_val = 1.0
T_val = 298.15

# Compute gradients
d_rec_d_stages = grad(nd_recovery, argnums=0)(n_stages_val, SF_ratio_val, T_val)
d_rec_d_SF = grad(nd_recovery, argnums=1)(n_stages_val, SF_ratio_val, T_val)
d_rec_d_T = grad(nd_recovery, argnums=2)(n_stages_val, SF_ratio_val, T_val)

print("Sensitivity Analysis for Nd Recovery")
print("=" * 50)

print(f"\n∂(Nd recovery)/∂(n_stages) = {float(d_rec_d_stages):.4f}")
print(f"  → Adding 1 stage increases recovery by {float(d_rec_d_stages)*100:.2f}%")

print(f"\n∂(Nd recovery)/∂(S/F ratio) = {float(d_rec_d_SF):.4f}")
print(f"  → 10% more solvent increases recovery by {float(d_rec_d_SF)*0.1*100:.2f}%")

print(f"\n∂(Nd recovery)/∂T = {float(d_rec_d_T):.6f} K⁻¹")
print(f"  → 10K increase changes recovery by {float(d_rec_d_T)*10*100:.2f}%")

## 6. Optimization: Maximize Nd Purity

**Goal**: Maximize Nd purity in extract (mole fraction among REEs)

This is a common objective when producing high-grade Nd for magnets.

In [ ]:
def nd_purity(params_arr):
    """Nd purity in extract (mole fraction among REEs)."""
    n_stages, S_F_ratio, T = params_arr
    
    solvent_adj = make_stream(
        flows={"Organic": 10.0 * S_F_ratio, "La": 0.0, "Nd": 0.0, "Dy": 0.0},
        T=T, P=101325.0,
    )
    
    params = CascadeParams(n_stages=n_stages, equilibrium=lle_eq, flow_config="counter_current")
    cascade_fn = MultistageCascade(params)
    
    _, extract, _ = cascade_fn(feed, solvent_adj, T=T)
    ext_flows = get_flows(extract)
    
    total_REE = ext_flows['La'] + ext_flows['Nd'] + ext_flows['Dy']
    return ext_flows['Nd'] / (total_REE + 1e-10)


def neg_nd_purity(params_arr):
    return -nd_purity(params_arr)


# Gradient descent optimization
params = jnp.array([5.0, 1.0, 298.15])
learning_rates = jnp.array([0.5, 0.01, 1.0])

print("Optimizing Nd Purity in Extract")
print("=" * 50)
print(f"\nInitial: n_stages={params[0]:.1f}, S/F={params[1]:.2f}, T={params[2]:.1f}K")
print(f"Initial Nd purity: {float(nd_purity(params))*100:.2f}%")

print("\nOptimization progress:")
for i in range(50):
    grads = grad(neg_nd_purity)(params)
    params = params - learning_rates * grads
    
    # Bounds
    params = jnp.array([
        jnp.clip(params[0], 2.0, 15.0),
        jnp.clip(params[1], 0.5, 3.0),
        jnp.clip(params[2], 280.0, 350.0),
    ])
    
    if (i + 1) % 10 == 0:
        purity = nd_purity(params)
        print(f"  Iter {i+1}: n={params[0]:.1f}, S/F={params[1]:.2f}, T={params[2]:.1f}K, purity={float(purity)*100:.2f}%")

print(f"\n✓ Optimized:")
print(f"  n_stages = {float(params[0]):.1f}")
print(f"  S/F ratio = {float(params[1]):.2f}")
print(f"  T = {float(params[2]):.1f} K")
print(f"  Nd purity = {float(nd_purity(params))*100:.2f}%")

## 7. Trade-off Analysis: Recovery vs Purity

There's an inherent trade-off:
- **More solvent** → Higher Nd recovery, but lower purity (more La extracted)
- **Less solvent** → Lower recovery, but higher purity

In [ ]:
print("Recovery vs Purity Trade-off (varying S/F ratio)")
print("=" * 60)
print(f"{'S/F':>6} {'Nd Rec%':>10} {'Nd Purity%':>12} {'La Rec%':>10} {'Dy Rec%':>10}")
print("-" * 60)

for sf in [0.5, 0.75, 1.0, 1.5, 2.0, 3.0]:
    solvent_adj = make_stream(
        flows={"Organic": 10.0 * sf, "La": 0.0, "Nd": 0.0, "Dy": 0.0},
        T=298.15, P=101325.0,
    )
    
    params = CascadeParams(n_stages=5, equilibrium=lle_eq, flow_config="counter_current")
    cascade_fn = MultistageCascade(params)
    
    _, extract, _ = cascade_fn(feed, solvent_adj, T=298.15)
    ext_flows = get_flows(extract)
    
    nd_rec = float(ext_flows['Nd']) / feed_flows['Nd'] * 100
    la_rec = float(ext_flows['La']) / feed_flows['La'] * 100
    dy_rec = float(ext_flows['Dy']) / feed_flows['Dy'] * 100
    
    total_REE = float(ext_flows['La'] + ext_flows['Nd'] + ext_flows['Dy'])
    nd_pur = float(ext_flows['Nd']) / total_REE * 100 if total_REE > 0 else 0
    
    print(f"{sf:>6.2f} {nd_rec:>10.1f} {nd_pur:>12.1f} {la_rec:>10.1f} {dy_rec:>10.1f}")

print("\n📊 Key insight: Higher S/F increases recovery but decreases purity")

## 8. Jacobian Analysis: Full Sensitivity Matrix

The **Jacobian** shows how all recoveries depend on all parameters simultaneously.

In [ ]:
def all_recoveries(params_arr):
    n_stages, S_F_ratio, T = params_arr
    
    solvent_adj = make_stream(
        flows={"Organic": 10.0 * S_F_ratio, "La": 0.0, "Nd": 0.0, "Dy": 0.0},
        T=T, P=101325.0,
    )
    
    params = CascadeParams(n_stages=n_stages, equilibrium=lle_eq, flow_config="counter_current")
    cascade_fn = MultistageCascade(params)
    
    _, extract, _ = cascade_fn(feed, solvent_adj, T=T)
    ext_flows = get_flows(extract)
    
    return jnp.array([
        ext_flows['La'] / feed_flows['La'],
        ext_flows['Nd'] / feed_flows['Nd'],
        ext_flows['Dy'] / feed_flows['Dy'],
    ])


params_eval = jnp.array([5.0, 1.0, 298.15])
J = jacfwd(all_recoveries)(params_eval)

print("Jacobian Matrix: ∂(recoveries)/∂(parameters)")
print("=" * 55)
print("                  n_stages      S/F ratio          T")
print(f"  ∂(La rec)     {J[0,0]:10.4f}   {J[0,1]:10.4f}   {J[0,2]:10.6f}")
print(f"  ∂(Nd rec)     {J[1,0]:10.4f}   {J[1,1]:10.4f}   {J[1,2]:10.6f}")
print(f"  ∂(Dy rec)     {J[2,0]:10.4f}   {J[2,1]:10.4f}   {J[2,2]:10.6f}")

print("\n📊 Interpretation:")
print(f"  • Dy is most sensitive to n_stages (∂Dy/∂n = {J[2,0]:.4f})")
print(f"  • Nd benefits most from more solvent (∂Nd/∂(S/F) = {J[1,1]:.4f})")
print(f"  • Temperature effects are negative (extraction is exothermic)")

## Summary

This notebook demonstrated:

1. **LLE fundamentals** - Distribution coefficients, separation factors
2. **Multi-stage extraction** - Counter-current cascade with Kremser equation
3. **Sensitivity analysis** - Exact gradients via automatic differentiation
4. **Optimization** - Maximize Nd purity using gradient descent
5. **Trade-off analysis** - Recovery vs purity
6. **Jacobian analysis** - Full input-output sensitivities

**Key advantages of differentiable LLE simulation:**
- Rapid optimization of extraction conditions
- Sensitivity analysis for process design
- Continuous relaxation of discrete variables (n_stages)